In [2]:
#imports
import pandas as pd
import numpy as np


In [57]:
### Men's Massey Ordinals

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#pre tournament rankings
pre_tournament_massey_m = massey_ordinals_m.query("RankingDayNum == 133")

#good subset of rankings
good_ratings_m = ["POM", "EBP", "MAS", "TRK", "HAS"]
g_pre_tournament_massey_m = pre_tournament_massey_m.query("Season >= 2016").loc[lambda df: df["SystemName"].isin(good_ratings_m)]

#group by and summarize
massey_m = g_pre_tournament_massey_m.groupby(["Season", "TeamID"]).agg(avg_massey_rank=("OrdinalRank", "mean")).reset_index()


In [58]:
### Men's AP Rankings

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#preseason rankings
preseason_ap_m = massey_ordinals_m.query("SystemName == 'AP'") 
preseason_ap_m = preseason_ap_m[preseason_ap_m['RankingDayNum'] == preseason_ap_m.groupby('Season')['RankingDayNum'].transform('min')]
preseason_ap_m = preseason_ap_m.rename({'OrdinalRank': 'ap_preseason_rank'}, axis='columns')
preseason_ap_m = preseason_ap_m[['Season', 'TeamID', 'ap_preseason_rank']]

#pretournament ratings
pre_tournament_ap_m = massey_ordinals_m.query("SystemName == 'AP' & RankingDayNum == 133")
pre_tournament_ap_m = pre_tournament_ap_m.rename({'OrdinalRank': 'ap_pretournament_rank'}, axis='columns')
pre_tournament_ap_m = pre_tournament_ap_m[['Season', 'TeamID', 'ap_pretournament_rank']]

#combine
ap_m = pd.merge(preseason_ap_m, pre_tournament_ap_m, how="outer", on=["Season", "TeamID"])


In [29]:
### Get Stats Function

def get_stats(reg_stats):

    #Remove Loc, which causes problems
    reg_stats = reg_stats.drop(columns = 'WLoc')

    w_reg_stats = reg_stats.copy()
    l_reg_stats = reg_stats.copy()

    #For games team is winner
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^L', 'opp_', regex=True)
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^W', '', regex=True)

    #For games team is loser
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^W', 'opp_', regex=True)
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^L', '', regex=True)

    #combine the data
    reg_stats_pg = pd.concat([w_reg_stats, l_reg_stats])

    #per game percentages (need for tempo adjusted average percentages)
    reg_stats_pg = (
        reg_stats_pg.assign(margin = reg_stats_pg['Score'] - reg_stats_pg['opp_Score'])
            .assign(poss = lambda x: x['FGA'] - x['OR'] + x['TO'] + 0.475*x['FTA'])
            .assign(opp_poss = lambda x: x['opp_FGA'] - x['opp_OR'] + x['opp_TO'] + 0.475*x['opp_FTA'])
            .assign(eff=lambda x: x['Score'] / x['poss'])
            .assign(opp_eff=lambda x: x['opp_Score'] / x['opp_poss'])

            .assign(fg_per=lambda x: x['FGM'] / x['FGA'])
            .assign(fg_a_per=lambda x: x['FGA'] / x['poss'])
            .assign(thr_a_per=lambda x: x['FGA3'] / x['poss'])
            .assign(to_per=lambda x: x['TO'] / x['poss'])
            .assign(blk_per=lambda x: x['Blk'] / x['opp_FGA'])
            .assign(foul_rec_per=lambda x: x['opp_PF'] / x['poss'])
            .assign(foul_per=lambda x: x['PF'] / x['opp_poss'])
            .assign(or_per=lambda x: x['OR'] / (x['OR'] + x['opp_DR']))
            .assign(dr_per=lambda x: x['DR'] / (x['DR'] + x['opp_OR']))
        
            .assign(opp_fg_a_per=lambda x: x['opp_FGA'] / x['opp_poss'])
            .assign(opp_fg_per=lambda x: x['opp_FGM'] / x['opp_FGA'])
            .assign(opp_to_per=lambda x: x['opp_TO'] / x['opp_poss'])
    )

    f_reg_stats = reg_stats_pg.groupby(["Season", "TeamID"]).agg(
        avg_score=("Score", "mean"),
        avg_opp_score=("opp_Score", "mean"),
        avg_margin = ("margin", "mean"),
        avg_poss = ("poss", "mean"),
        avg_eff = ("eff", "mean"),
        avg_opp_eff = ("opp_eff", "mean"),
        avg_fg_per=("fg_per", "mean"),
        avg_m3=("FGM3", "mean"),
        avg_a3=("FGA3", "mean"),#
        avg_ftm=("FTM", "mean"),
        avg_fta=("FTA", "mean"),#
        avg_fg_a_per=("fg_a_per", "mean"),
        avg_thr_a_per=("thr_a_per", "mean"),
        avg_to_per=("to_per", "mean"),
        avg_blk_per=("blk_per", "mean"),
        avg_foul_rec_per=("foul_rec_per", "mean"),
        avg_foul_per=("foul_per", "mean"),
        avg_or_per=("or_per", "mean"),
        avg_dr_per=("dr_per", "mean"),
        avg_opp_fg_a_per=("opp_fg_a_per", "mean"),
        avg_opp_fg_per=("opp_fg_per", "mean"),
        avg_opp_to_per=("opp_to_per", "mean")
    ).reset_index()

    f_reg_stats = (
        f_reg_stats.assign(avg_thr_per = f_reg_stats['avg_m3'] / f_reg_stats['avg_a3'])
        .assign(ft_per=lambda x: x['avg_ftm'] / x['avg_fta'])
    )

    f_reg_stats = f_reg_stats.drop(columns = ['avg_a3', 'avg_m3', 'avg_ftm', 'avg_fta'])

    return f_reg_stats

In [56]:
### Get Stats

#data
reg_stats_m = pd.read_csv("data_2026/MRegularSeasonDetailedResults.csv")
reg_stats_w = pd.read_csv("data_2026/WRegularSeasonDetailedResults.csv")

#men's
f_reg_stats_m = get_stats(reg_stats_m)

#women's
f_reg_stats_w = get_stats(reg_stats_w)


#f_reg_stats_m.sort_values('avg_eff', ascending = False).head()

In [38]:
### Seed
seed_m = pd.read_csv("data_2026/MNCAATourneySeeds.csv")
seed_w = pd.read_csv("data_2026/WNCAATourneySeeds.csv")

seed_m = seed_m.assign(Seed = seed_m['Seed'].str[1:3])
seed_w = seed_w.assign(Seed = seed_w['Seed'].str[1:3])

In [ ]:
### Combine Data

#Men's
combined_m = (
    seed_m
    .merge(ap_m, on=["Season", "TeamID"], how="outer")
    .merge(massey_m, on=["Season", "TeamID"], how="outer")
    .merge(f_reg_stats_m, on=["Season", "TeamID"], how="outer")
)

#Women's
combined_w = (
    seed_w
    .merge(f_reg_stats_w, on=["Season", "TeamID"], how="outer")
)

In [ ]:
### Get results

def get_results(tourney):
    tourney = (tourney.assign(
            teamA = tourney[['WTeamID','LTeamID']].min(axis=1),
            teamB = tourney[['WTeamID','LTeamID']].max(axis=1)
        ).assign(
            result = lambda x: (x['WTeamID'] == x['teamA']).astype(int),
            scoreA = lambda x: x['result']*x['WScore'] + (1 - x['result'])*x['LScore'],
            scoreB = lambda x: x['result']*x['LScore'] + (1 - x['result'])*x['WScore'],
            margin = lambda x: (x['scoreA'] - x['scoreB'])
        )
    )

    tourney = tourney[['Season', 'DayNum', 'teamA', 'teamB', 'scoreA', 'scoreB', 'margin', 'result', 'NumOT']]
    
    return tourney


#Men's
tourney_m = pd.read_csv("data_2026/MNCAATourneyCompactResults.csv")
tourney_results_m = get_results(tourney_m)

#Women's
tourney_w = pd.read_csv("data_2026/WNCAATourneyCompactResults.csv")
tourney_results_w = get_results(tourney_w)



,Season,DayNum,teamA,teamB,scoreA,scoreB,margin,result,NumOT
0,1998,137,3104,3422,94,46,48,1,0
1,1998,137,3112,3365,75,63,12,1,0
2,1998,137,3163,3193,93,52,41,1,0
3,1998,137,3198,3266,59,45,14,1,0
4,1998,137,3203,3208,74,72,2,1,0
5,1998,137,3234,3269,77,59,18,1,0
6,1998,137,3242,3408,72,68,4,1,0
7,1998,137,3263,3301,64,89,-25,0,0
8,1998,137,3304,3307,76,59,17,1,0
9,1998,137,3224,3314,71,91,-20,0,0
